# EPIC Clarity — Procedure Occurrence Hydration

Populates `_exponent.omop_epic.procedure_occurrence` from Epic Clarity procedure data.

## Source Tables
- `_exponent._bronze_epic_clarity.order_proc` — procedure orders (DESCRIPTION only)
- `_exponent._bronze_epic_clarity.or_log` — surgical log (SURGERY_DATE)
- `_exponent._bronze_epic_clarity.hsp_acct_cpt_codes` — **Hospital billing CPT codes** (NEW)

## Concept Mapping Strategy
- **hsp_acct_cpt_codes**: Join `CPT_CODE` → concept table for CPT4/HCPCS → Procedure concept mapping
- **order_proc/or_log**: Use DESCRIPTION as source_value (fallback when no CPT)

## Pipeline
1. `standard_concept_mapping` — CPT4/HCPCS → OMOP standard concept (Procedure domain)
2. `silver_procedure_occurrence` — staged temp view with billing CPT joins
3. MERGE → `omop_silver.procedure_occurrence`
4. INSERT → `omop_mapping.source_to_procedure_occurrence`
5. `gold` — resolves surrogate IDs and FK references
6. MERGE → `omop_epic.procedure_occurrence`

## Notes
- `order_proc.CPT_CODE` is unpopulated, so we use `hsp_acct_cpt_codes` as primary CPT source
- procedure_type_concept_id = 32817 (EHR) for order_proc/or_log, 32810 (Claim) for billing

In [ ]:
%sql
-- TRUNCATE Gold table for Epic (run this to clear stale data before reload)
TRUNCATE TABLE _exponent.omop_epic.procedure_occurrence;

In [ ]:
%sql
-- Delete Epic records from Silver and Mapping tables (for full refresh)
DELETE FROM _exponent.omop_silver.procedure_occurrence WHERE source_system = 'epic_clarity';

DELETE FROM _exponent.omop_mapping.source_to_procedure_occurrence WHERE source_system = 'epic_clarity';

# Transformation

In [ ]:
%sql
-- Create standard concept mapping view for CPT4/HCPCS → Procedure concepts
-- Maps CPT and HCPCS codes to standard Procedure domain concepts
CREATE OR REPLACE TEMPORARY VIEW standard_concept_mapping AS
WITH ranked AS (
  SELECT
    concept.vocabulary_id         AS source_vocabulary_id,
    concept.concept_code          AS source_concept_code,
    concept.concept_id            AS source_concept_id,
    standard_concept.concept_id   AS standard_concept_id,
    standard_concept.concept_name AS standard_concept_name,
    concept_relationship.valid_start_date AS rel_valid_start_date,
    ROW_NUMBER() OVER (
      PARTITION BY concept.vocabulary_id, concept.concept_code
      ORDER BY concept_relationship.valid_start_date DESC, standard_concept.concept_id ASC
    ) AS rn
  FROM _exponent.omop.concept
  JOIN _exponent.omop.concept_relationship
    ON concept_relationship.concept_id_1 = concept.concept_id
   AND concept_relationship.relationship_id = 'Maps to'
   AND concept_relationship.invalid_reason IS NULL
  JOIN _exponent.omop.concept AS standard_concept
    ON standard_concept.concept_id = concept_relationship.concept_id_2
   AND standard_concept.standard_concept = 'S'
   AND standard_concept.domain_id = 'Procedure'
   AND standard_concept.invalid_reason IS NULL
  WHERE concept.vocabulary_id IN ('CPT4', 'HCPCS')
    AND concept.invalid_reason IS NULL
)
SELECT
  source_vocabulary_id,
  source_concept_code,
  source_concept_id,
  standard_concept_id,
  standard_concept_name
FROM ranked
WHERE rn = 1;

In [ ]:
%sql
-- Create silver_procedure_occurrence temp view for Epic Clarity
-- Now includes billing CPT codes with concept mapping
CREATE OR REPLACE TEMPORARY VIEW silver_procedure_occurrence AS

-- =========================================================
-- 1. Hospital Billing CPT Codes (PRIMARY SOURCE for mapped procedures)
-- =========================================================
SELECT
  stp.person_id,
  -- Map CPT code to standard procedure concept
  COALESCE(scm.standard_concept_id, 0) AS procedure_concept_id,
  DATE(cpt.CPT_CODE_DATE) AS procedure_date,
  cpt.CPT_CODE_DATE AS procedure_datetime,
  32810 AS procedure_type_concept_id,  -- Claim (billing source)
  0 AS modifier_concept_id,
  CAST(cpt.CPT_QUANTITY AS DOUBLE) AS quantity,
  NULL AS provider_id,
  NULL AS visit_occurrence_id,
  NULL AS visit_detail_id,
  -- Store CPT code as procedure_source_value
  cpt.CPT_CODE AS procedure_source_value,
  -- Source concept is the CPT concept itself
  COALESCE(scm.source_concept_id, 0) AS procedure_source_concept_id,
  cpt.CPT_MODIFIERS AS modifier_source_value,
  CONCAT_WS(CHR(31), 'epic_clarity', 'hsp_acct_cpt_codes', 'HSP_ACCOUNT_ID', CAST(cpt.HSP_ACCOUNT_ID AS STRING), 'LINE', CAST(cpt.LINE AS STRING)) AS procedure_occurrence_source_value,
  'epic_clarity' AS source_system
FROM _exponent._bronze_epic_clarity.hsp_acct_cpt_codes cpt
-- Join to hsp_account to get PAT_ID
INNER JOIN _exponent._bronze_epic_clarity.hsp_account ha
  ON cpt.HSP_ACCOUNT_ID = ha.HSP_ACCOUNT_ID
-- Join to person mapping
INNER JOIN _exponent.omop_mapping.source_to_person stp
  ON stp.person_source_value = CONCAT_WS(CHR(31), 'epic_clarity', 'PATIENT', 'PAT_ID', ha.PAT_ID)
  AND stp.active_flag = TRUE
-- Map CPT code to standard procedure concept
LEFT JOIN standard_concept_mapping scm
  ON TRIM(cpt.CPT_CODE) = scm.source_concept_code
  AND scm.source_vocabulary_id = 'CPT4'
WHERE cpt.CPT_CODE IS NOT NULL
  AND cpt.CPT_CODE_DATE IS NOT NULL
  AND ha.PAT_ID IS NOT NULL

UNION ALL

-- =========================================================
-- 2. Procedure Orders (completed, non-lab) - no CPT codes available
-- =========================================================
SELECT
  stp.person_id,
  0 AS procedure_concept_id,  -- No CPT codes available in order_proc
  DATE(COALESCE(op.PROC_DATE, op.ORDERING_DATE)) AS procedure_date,
  COALESCE(op.PROC_DATE, op.ORDERING_DATE) AS procedure_datetime,
  32817 AS procedure_type_concept_id,  -- EHR
  0 AS modifier_concept_id,
  CAST(op.QUANTITY AS DOUBLE) AS quantity,
  NULL AS provider_id,
  NULL AS visit_occurrence_id,
  NULL AS visit_detail_id,
  op.DESCRIPTION AS procedure_source_value,
  0 AS procedure_source_concept_id,
  NULL AS modifier_source_value,
  CONCAT_WS(CHR(31), 'epic_clarity', 'order_proc', 'ORDER_PROC_ID', CAST(op.ORDER_PROC_ID AS STRING)) AS procedure_occurrence_source_value,
  'epic_clarity' AS source_system
FROM _exponent._bronze_epic_clarity.order_proc op
INNER JOIN _exponent.omop_mapping.source_to_person stp
  ON stp.person_source_value = CONCAT_WS(CHR(31), 'epic_clarity', 'PATIENT', 'PAT_ID', op.PAT_ID)
  AND stp.active_flag = TRUE
WHERE op.ORDER_STATUS_C = 5  -- Completed
  AND op.ORDER_TYPE_C != 7   -- Exclude labs
  AND op.PAT_ID IS NOT NULL
  AND COALESCE(op.PROC_DATE, op.ORDERING_DATE) IS NOT NULL

UNION ALL

-- =========================================================
-- 3. Surgical Log - no CPT codes available
-- =========================================================
SELECT
  stp.person_id,
  0 AS procedure_concept_id,
  DATE(ol.SURGERY_DATE) AS procedure_date,
  ol.SURGERY_DATE AS procedure_datetime,
  32817 AS procedure_type_concept_id,  -- EHR
  0 AS modifier_concept_id,
  NULL AS quantity,
  NULL AS provider_id,
  NULL AS visit_occurrence_id,
  NULL AS visit_detail_id,
  COALESCE(ol.LOG_NAME, 'Surgical Procedure') AS procedure_source_value,
  0 AS procedure_source_concept_id,
  NULL AS modifier_source_value,
  CONCAT_WS(CHR(31), 'epic_clarity', 'or_log', 'LOG_ID', CAST(ol.LOG_ID AS STRING)) AS procedure_occurrence_source_value,
  'epic_clarity' AS source_system
FROM _exponent._bronze_epic_clarity.or_log ol
INNER JOIN _exponent.omop_mapping.source_to_person stp
  ON stp.person_source_value = CONCAT_WS(CHR(31), 'epic_clarity', 'PATIENT', 'PAT_ID', ol.PAT_ID)
  AND stp.active_flag = TRUE
WHERE ol.PAT_ID IS NOT NULL
  AND ol.SURGERY_DATE IS NOT NULL
  AND (ol.STATUS_C IS NULL OR ol.STATUS_C != 10);  -- Exclude voided

In [0]:
# %sql
# -- Preview
# SELECT * FROM silver_procedure_occurrence LIMIT 10

In [0]:
# %sql
# -- Check for duplicates on merge key
# SELECT procedure_occurrence_source_value, COUNT(*) AS cnt
# FROM silver_procedure_occurrence
# GROUP BY procedure_occurrence_source_value
# HAVING COUNT(*) > 1
# LIMIT 10

# Write to Silver

In [ ]:
%sql
-- Merge to Silver layer
MERGE INTO _exponent.omop_silver.procedure_occurrence AS t
USING (
  SELECT * FROM (
    SELECT *,
      ROW_NUMBER() OVER (
        PARTITION BY procedure_occurrence_source_value
        ORDER BY procedure_date DESC
      ) AS rn
    FROM silver_procedure_occurrence
  ) WHERE rn = 1
) AS s
ON t.procedure_occurrence_source_value = s.procedure_occurrence_source_value

WHEN MATCHED AND (
     NOT (t.person_id <=> s.person_id)
  OR NOT (t.procedure_concept_id <=> s.procedure_concept_id)
  OR NOT (t.procedure_date <=> s.procedure_date)
  OR NOT (t.procedure_datetime <=> s.procedure_datetime)
  OR NOT (t.procedure_type_concept_id <=> s.procedure_type_concept_id)
  OR NOT (t.procedure_source_value <=> s.procedure_source_value)
  OR NOT (t.procedure_source_concept_id <=> s.procedure_source_concept_id)
  OR NOT (t.modifier_source_value <=> s.modifier_source_value)
  OR NOT (t.source_system <=> s.source_system)
)
THEN UPDATE SET
  t.person_id                     = s.person_id,
  t.procedure_concept_id          = s.procedure_concept_id,
  t.procedure_date                = s.procedure_date,
  t.procedure_datetime            = s.procedure_datetime,
  t.procedure_type_concept_id     = s.procedure_type_concept_id,
  t.modifier_concept_id           = s.modifier_concept_id,
  t.quantity                      = s.quantity,
  t.provider_id                   = s.provider_id,
  t.visit_occurrence_id           = s.visit_occurrence_id,
  t.visit_detail_id               = s.visit_detail_id,
  t.procedure_source_value        = s.procedure_source_value,
  t.procedure_source_concept_id   = s.procedure_source_concept_id,
  t.modifier_source_value         = s.modifier_source_value,
  t.source_system                 = s.source_system,
  t.last_mod_tsp                  = current_timestamp()

WHEN NOT MATCHED THEN INSERT (
  procedure_occurrence_source_value,
  person_id,
  procedure_concept_id,
  procedure_date,
  procedure_datetime,
  procedure_type_concept_id,
  modifier_concept_id,
  quantity,
  provider_id,
  visit_occurrence_id,
  visit_detail_id,
  procedure_source_value,
  procedure_source_concept_id,
  modifier_source_value,
  source_system,
  last_mod_tsp
)
VALUES (
  s.procedure_occurrence_source_value,
  s.person_id,
  s.procedure_concept_id,
  s.procedure_date,
  s.procedure_datetime,
  s.procedure_type_concept_id,
  s.modifier_concept_id,
  s.quantity,
  s.provider_id,
  s.visit_occurrence_id,
  s.visit_detail_id,
  s.procedure_source_value,
  s.procedure_source_concept_id,
  s.modifier_source_value,
  s.source_system,
  current_timestamp()
);

In [0]:
# %sql
# -- Verify silver
# SELECT * FROM _exponent.omop_silver.procedure_occurrence
# WHERE source_system = 'epic_clarity'
# LIMIT 10

# Register Procedure Occurrence IDs

In [0]:
%sql
INSERT INTO _exponent.omop_mapping.source_to_procedure_occurrence (
    source_system,
    procedure_occurrence_source_value,
    active_flag,
    created_tsp,
    last_mod_tsp
)
SELECT
    s.source_system,
    s.procedure_occurrence_source_value,
    TRUE AS active_flag,
    current_timestamp() AS created_tsp,
    COALESCE(s.last_mod_tsp, current_timestamp()) AS last_mod_tsp
FROM (
    SELECT DISTINCT source_system, procedure_occurrence_source_value, last_mod_tsp
    FROM _exponent.omop_silver.procedure_occurrence
    WHERE source_system = 'epic_clarity'
) s
LEFT ANTI JOIN _exponent.omop_mapping.source_to_procedure_occurrence x
  ON s.procedure_occurrence_source_value = x.procedure_occurrence_source_value;

# Write to Gold

In [ ]:
%sql
-- Merge to Gold layer
-- MERGE INTO _exponent.omop.procedure_occurrence AS gold
MERGE INTO _exponent.omop_epic.procedure_occurrence AS gold
USING (
  SELECT
    spo.procedure_occurrence_id,
    s.person_id,
    s.procedure_concept_id,
    s.procedure_date,
    s.procedure_datetime,
    s.procedure_type_concept_id,
    s.modifier_concept_id,
    s.quantity,
    s.provider_id,
    s.visit_occurrence_id,
    s.visit_detail_id,
    s.procedure_source_value,
    s.procedure_source_concept_id,
    s.modifier_source_value
  FROM _exponent.omop_silver.procedure_occurrence s
  JOIN _exponent.omop_mapping.source_to_procedure_occurrence spo
    ON spo.procedure_occurrence_source_value = s.procedure_occurrence_source_value
   AND spo.active_flag = TRUE
  -- Ensure person_id exists in gold person table (prevents orphan records)
  INNER JOIN _exponent.omop_epic.person p
    ON p.person_id = s.person_id
  WHERE s.source_system = 'epic_clarity'
    AND s.person_id IS NOT NULL
    -- Exclude procedures before birth (plausibility check)
    AND s.procedure_date >= DATE(p.birth_datetime)
) AS src
ON gold.procedure_occurrence_id = src.procedure_occurrence_id

WHEN MATCHED THEN UPDATE SET
  gold.person_id                      = src.person_id,
  gold.procedure_concept_id           = src.procedure_concept_id,
  gold.procedure_date                 = src.procedure_date,
  gold.procedure_datetime             = src.procedure_datetime,
  gold.procedure_type_concept_id      = src.procedure_type_concept_id,
  gold.modifier_concept_id            = src.modifier_concept_id,
  gold.quantity                       = src.quantity,
  gold.provider_id                    = src.provider_id,
  gold.visit_occurrence_id            = src.visit_occurrence_id,
  gold.visit_detail_id                = src.visit_detail_id,
  gold.procedure_source_value         = src.procedure_source_value,
  gold.procedure_source_concept_id    = src.procedure_source_concept_id,
  gold.modifier_source_value          = src.modifier_source_value

WHEN NOT MATCHED THEN INSERT (
  procedure_occurrence_id,
  person_id,
  procedure_concept_id,
  procedure_date,
  procedure_datetime,
  procedure_type_concept_id,
  modifier_concept_id,
  quantity,
  provider_id,
  visit_occurrence_id,
  visit_detail_id,
  procedure_source_value,
  procedure_source_concept_id,
  modifier_source_value
)
VALUES (
  src.procedure_occurrence_id,
  src.person_id,
  src.procedure_concept_id,
  src.procedure_date,
  src.procedure_datetime,
  src.procedure_type_concept_id,
  src.modifier_concept_id,
  src.quantity,
  src.provider_id,
  src.visit_occurrence_id,
  src.visit_detail_id,
  src.procedure_source_value,
  src.procedure_source_concept_id,
  src.modifier_source_value
);

# Validation

In [0]:
# %sql
# -- Layer counts
# SELECT 'Silver' AS layer, COUNT(*) AS record_count FROM _exponent.omop_silver.procedure_occurrence WHERE source_system = 'epic_clarity'
# UNION ALL
# SELECT 'Mapping' AS layer, COUNT(*) AS record_count FROM _exponent.omop_mapping.source_to_procedure_occurrence WHERE source_system = 'epic_clarity'
# UNION ALL
# SELECT 'Gold' AS layer, COUNT(*) AS record_count FROM _exponent.omop.procedure_occurrence

In [0]:
# %sql
# -- Verify gold
# SELECT * FROM _exponent.omop.procedure_occurrence
# LIMIT 10

In [0]:
# %sql
# -- Top procedure descriptions
# SELECT procedure_source_value, COUNT(*) AS cnt
# FROM _exponent.omop.procedure_occurrence
# GROUP BY procedure_source_value
# ORDER BY cnt DESC
# LIMIT 20

In [ ]:
%sql
-- Validation queries
-- 1. Check person_id FK integrity (should be 0 orphan records)
SELECT 'PROCEDURE_OCCURRENCE.PERSON_ID FK' as check_field,
       COUNT(*) as orphan_records
FROM _exponent.omop_epic.procedure_occurrence po
WHERE NOT EXISTS (SELECT 1 FROM _exponent.omop_epic.person p WHERE p.person_id = po.person_id);

-- 2. Total record count
SELECT 'total_records' as check_field, COUNT(*) as cnt FROM _exponent.omop_epic.procedure_occurrence;

-- 3. Procedure concept mapping rate (should be > 0% now with billing CPT)
SELECT 'procedure_concept_id mapping' as check_field,
       COUNT(*) as total,
       SUM(CASE WHEN procedure_concept_id = 0 THEN 1 ELSE 0 END) as unmapped,
       SUM(CASE WHEN procedure_concept_id > 0 THEN 1 ELSE 0 END) as mapped,
       ROUND(SUM(CASE WHEN procedure_concept_id > 0 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) as pct_mapped
FROM _exponent.omop_epic.procedure_occurrence;

-- 4. Records by type (EHR vs Claim)
SELECT 'procedure_type_concept_id distribution' as check_field,
       procedure_type_concept_id,
       CASE procedure_type_concept_id 
         WHEN 32817 THEN 'EHR'
         WHEN 32810 THEN 'Claim'
         ELSE 'Other'
       END as type_name,
       COUNT(*) as cnt
FROM _exponent.omop_epic.procedure_occurrence
GROUP BY procedure_type_concept_id
ORDER BY cnt DESC;

-- 5. Top 10 unmapped CPT codes (for future mapping work)
SELECT procedure_source_value as cpt_code, COUNT(*) as cnt
FROM _exponent.omop_epic.procedure_occurrence
WHERE procedure_concept_id = 0
  AND procedure_type_concept_id = 32810  -- Only billing records that should have CPT
GROUP BY procedure_source_value
ORDER BY cnt DESC
LIMIT 10;

-- 6. Top 10 mapped procedure concepts
SELECT procedure_concept_id, c.concept_name, COUNT(*) as cnt
FROM _exponent.omop_epic.procedure_occurrence po
LEFT JOIN _exponent.omop.concept c ON c.concept_id = po.procedure_concept_id
WHERE procedure_concept_id > 0
GROUP BY procedure_concept_id, c.concept_name
ORDER BY cnt DESC
LIMIT 10;